# 2.5 - Final Dataset Preparation

**Taller de Programación - UBA FCE | Grupo JLP**

---

## Objetivo

Preparar el dataset final limpio para modelado:

1. **Cargar features completos** (step4 con clima)
2. **Limpieza consolidada de NaNs** (evitar repetir en cada modelo)
3. **Validación de calidad** (verificar consistencia temporal)
4. **Guardar dataset listo para modelado** con metadata completa

**Salida:**
- `features_final_modeling.csv` - Dataset limpio sin NaNs
- `metadata_final_dataset.json` - Documentación completa de limpieza

## Setup

In [1]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import json
from datetime import datetime

# Agregar src al path
BASE_DIR = Path.cwd().parents[1]
sys.path.append(str(BASE_DIR / 'src'))

from config import PROCESSED_DIR, START_DATE, END_DATE, logger

# Configurar pandas display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print(f"✓ Base directory: {BASE_DIR}")
print(f"✓ Processed directory: {PROCESSED_DIR}")
print(f"✓ Período de análisis: {START_DATE} → {END_DATE}")

✓ Base directory: c:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal
✓ Processed directory: C:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\data\processed
✓ Período de análisis: 2000-01-01 → 2025-12-06


## 1. Cargar Dataset Base + Todas las Features Académicas

**Datasets a mergear:**
1. `features_step4_climate.csv` - Base (3,186 features)
2. `cftc/cftc_features_2000_2025.csv` - CFTC (+11 features)
3. `gdelt/sentiment_features_2000_2025.csv` - Sentiment (+10 features)
4. `bdi/bdi_features.csv` - Baltic Dry Index (+8 features)
5. `supply_demand/crop_conditions_all_features.csv` - Crop (+15 features)
6. `supply_demand/government_stocks_ers_all_features.csv` - Gov Stocks (+9 features)

**Total esperado:** 3,239 features

In [2]:
# Cargar dataset base (Step 4 - Climate)
df_base = pd.read_csv(PROCESSED_DIR / 'features_step4_climate.csv', index_col=0, parse_dates=True)

print("=" * 80)
print("DATASET BASE (Step 4 - Climate)")
print("=" * 80)
print(f"Shape: {df_base.shape}")
print(f"Período: {df_base.index.min()} → {df_base.index.max()}")
print(f"Features: {len(df_base.columns):,}")

# Cargar CFTC features
EXTERNAL_DIR = BASE_DIR / 'data' / 'external'
INTERIM_DIR = BASE_DIR / 'data' / 'interim'

cftc_path = EXTERNAL_DIR / 'cftc' / 'cftc_features_2000_2025.csv'
if cftc_path.exists():
    df_cftc = pd.read_csv(cftc_path, index_col=0, parse_dates=True)
    print(f"\n✓ CFTC: {df_cftc.shape} ({df_cftc.index.min()} → {df_cftc.index.max()})")
else:
    print(f"\n❌ CFTC no encontrado: {cftc_path}")
    df_cftc = None

# Cargar GDELT features
gdelt_path = EXTERNAL_DIR / 'gdelt' / 'sentiment_features_2000_2025.csv'
if gdelt_path.exists():
    df_gdelt = pd.read_csv(gdelt_path, index_col=0, parse_dates=True)
    print(f"✓ GDELT: {df_gdelt.shape} ({df_gdelt.index.min()} → {df_gdelt.index.max()})")
else:
    print(f"❌ GDELT no encontrado: {gdelt_path}")
    df_gdelt = None

# Cargar BDI features
bdi_path = INTERIM_DIR / 'predictors' / 'bdi_features.csv'
if bdi_path.exists():
    df_bdi = pd.read_csv(bdi_path, index_col=0, parse_dates=True)
    print(f"✓ BDI: {df_bdi.shape} ({df_bdi.index.min()} → {df_bdi.index.max()})")
else:
    print(f"❌ BDI no encontrado: {bdi_path}")
    df_bdi = None

# Cargar Crop Conditions
crop_path = INTERIM_DIR / 'supply_demand' / 'crop_conditions_all_features.csv'
if crop_path.exists():
    df_crop = pd.read_csv(crop_path, index_col=0, parse_dates=True)
    print(f"✓ Crop Conditions: {df_crop.shape} ({df_crop.index.min()} → {df_crop.index.max()})")
else:
    print(f"❌ Crop Conditions no encontrado: {crop_path}")
    df_crop = None

# Cargar Government Stocks
stocks_path = INTERIM_DIR / 'supply_demand' / 'government_stocks_ers_all_features.csv'
if stocks_path.exists():
    df_stocks = pd.read_csv(stocks_path, index_col=0, parse_dates=True)
    print(f"✓ Gov Stocks: {df_stocks.shape} ({df_stocks.index.min()} → {df_stocks.index.max()})")
else:
    print(f"❌ Gov Stocks no encontrado: {stocks_path}")
    df_stocks = None

DATASET BASE (Step 4 - Climate)
Shape: (6731, 3186)
Período: 2000-01-03 00:00:00 → 2025-11-10 00:00:00
Features: 3,186

✓ CFTC: (28290, 18) (2000-01-04 00:00:00 → 2025-10-28 00:00:00)
❌ GDELT no encontrado: c:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\data\external\gdelt\sentiment_features_2000_2025.csv
✓ BDI: (6456, 8) (2000-01-04 00:00:00 → 2025-11-07 00:00:00)
✓ Crop Conditions: (337, 15) (2024-10-27 00:00:00 → 2025-09-28 00:00:00)
✓ Gov Stocks: (23834, 9) (1960-05-31 00:00:00 → 2025-08-31 00:00:00)


In [3]:
# Merge todas las features (left join para mantener todas las fechas del base)
df_features = df_base.copy()

# Merge CFTC
if df_cftc is not None:
    df_features = df_features.join(df_cftc, how='left', rsuffix='_cftc')
    print(f"\n✓ Merged CFTC: {df_features.shape}")

# Merge GDELT
if df_gdelt is not None:
    df_features = df_features.join(df_gdelt, how='left', rsuffix='_gdelt')
    print(f"✓ Merged GDELT: {df_features.shape}")

# Merge BDI
if df_bdi is not None:
    df_features = df_features.join(df_bdi, how='left', rsuffix='_bdi')
    print(f"✓ Merged BDI: {df_features.shape}")

# Merge Crop Conditions
if df_crop is not None:
    df_features = df_features.join(df_crop, how='left', rsuffix='_crop')
    print(f"✓ Merged Crop: {df_features.shape}")

# Merge Government Stocks
if df_stocks is not None:
    df_features = df_features.join(df_stocks, how='left', rsuffix='_stocks')
    print(f"✓ Merged Gov Stocks: {df_features.shape}")

print("\n" + "=" * 80)
print("DATASET COMPLETO CON TODAS LAS FEATURES")
print("=" * 80)
print(f"Shape final: {df_features.shape}")
print(f"Período: {df_features.index.min()} → {df_features.index.max()}")
print(f"Total features: {len(df_features.columns):,}")
print(f"\nFeatures esperadas: 3,239")
print(f"Features obtenidas: {len(df_features.columns):,}")


✓ Merged CFTC: (20173, 3204)
✓ Merged BDI: (20173, 3212)
✓ Merged Crop: (20173, 3227)
✓ Merged Gov Stocks: (20173, 3236)

DATASET COMPLETO CON TODAS LAS FEATURES
Shape final: (20173, 3236)
Período: 2000-01-03 00:00:00 → 2025-11-10 00:00:00
Total features: 3,236

Features esperadas: 3,239
Features obtenidas: 3,236


## 1.1 Merge Todas las Features con Left Join

## 2. Diagnóstico de Missing Values

Antes de limpiar, documentar estado inicial de NaNs por tipo de feature.

In [4]:
# Calcular missing values por columna
missing_summary = pd.DataFrame({
    'missing_count': df_features.isnull().sum(),
    'missing_pct': (df_features.isnull().sum() / len(df_features) * 100).round(2)
}).sort_values('missing_count', ascending=False)

missing_summary = missing_summary[missing_summary['missing_count'] > 0]

print("=" * 80)
print("MISSING VALUES POR FEATURE")
print("=" * 80)
print(f"\nTotal features con missing: {len(missing_summary)} / {len(df_features.columns)}")
print(f"Total missing values: {df_features.isnull().sum().sum():,}")
print(f"Porcentaje total: {(df_features.isnull().sum().sum() / df_features.size * 100):.2f}%\n")

if len(missing_summary) > 0:
    print("Top 20 features con más missing:")
    print(missing_summary.head(20))

MISSING VALUES POR FEATURE

Total features con missing: 2840 / 3236
Total missing values: 5,363,614
Porcentaje total: 8.22%

Top 20 features con más missing:
                                         missing_count  missing_pct
Heat_Stress_Days_price_to_ma7                    20173        100.0
Baltic_Dry_Index_volume_price_to_ma30            20173        100.0
Baltic_Dry_Index_volume_price_to_ma7             20173        100.0
Heat_Stress_Days_price_to_ma90                   20173        100.0
Baltic_Dry_Index_volume_price_to_ma90            20173        100.0
Heat_Stress_Days_vol_ratio_30_90                 20173        100.0
Heat_Stress_Days_price_to_ma30                   20173        100.0
Baltic_Dry_Index_volume_log_return1              20173        100.0
Heat_Stress_Days_vol_ratio_7_30                  20173        100.0
Heat_Stress_Days_simple_return30                 20173        100.0
Heat_Stress_Days_log_return90                    20173        100.0
Baltic_Dry_Index_volume_vo

In [5]:
# Clasificar missing por tipo de feature
def clasificar_feature(col_name):
    """Clasificar feature por su nombre para análisis de missing."""
    if '_lag_' in col_name:
        return 'Temporal Lags'
    elif '_roll_' in col_name or '_ma_' in col_name or '_ema_' in col_name:
        return 'Rolling Stats'
    elif '_return_' in col_name or '_vol_' in col_name or '_bb_' in col_name:
        return 'Returns & Volatility'
    elif col_name.startswith('temp_') or col_name.startswith('prec_') or col_name in ['oni', 'et0_global', 'gdd_30d_global', 'heat_stress_days_30d', 'prec_deficit_30d']:
        return 'Climate'
    elif col_name.startswith('cftc_'):
        return 'CFTC Sentiment'
    elif col_name.startswith('gdelt_') or col_name.startswith('sentiment_'):
        return 'GDELT Sentiment'
    elif col_name.startswith('bdi_'):
        return 'Baltic Dry Index'
    elif col_name.startswith('crop_'):
        return 'Crop Conditions'
    elif col_name.startswith('gov_stocks_'):
        return 'Government Stocks'
    else:
        return 'Base Features'

# Agrupar missing por tipo
missing_summary['feature_type'] = missing_summary.index.map(clasificar_feature)
missing_by_type = missing_summary.groupby('feature_type').agg({
    'missing_count': ['sum', 'mean', 'count'],
    'missing_pct': 'mean'
}).round(2)

print("\n" + "=" * 80)
print("MISSING VALUES POR TIPO DE FEATURE")
print("=" * 80)
print(missing_by_type)


MISSING VALUES POR TIPO DE FEATURE
                     missing_count                missing_pct
                               sum     mean count        mean
feature_type                                                 
Baltic Dry Index              3344   836.00     4        4.14
Base Features              4011793  1957.93  2049        9.71
Returns & Volatility       1345633  1716.37   784        8.51
Temporal Lags                 2844   948.00     3        4.70


## 3. Estrategia de Limpieza de NaNs

**Principios:**
1. **Temporal Lags & Rolling Stats:** Forward fill (ffill) - usar valor anterior
2. **Returns & Volatility:** Median imputation (volatilidad histórica)
3. **Climate Features:** Median imputation (promedio climático)
4. **CFTC/GDELT/BDI:** Forward fill (sentiment se mantiene hasta nuevo dato)
5. **Crop/Gov Stocks:** Forward fill (datos mensuales/trimestrales)
6. **Base Features:** Ya no deberían tener missing

**Gap 2014 GDELT:**
- GDELT 1.0: 2000-2013
- GDELT 2.0: 2015-2025
- 2014: Forward fill desde 2013 (asumir sentiment estable)

**Orden de aplicación:**
1. ffill para features temporales (lags, rolling, CFTC, GDELT, BDI, Crop, Stocks)
2. Median imputation para features calculados (returns, volatility, clima)
3. Verificación final (assert no quedan NaNs)

In [6]:
 # Crear copia para limpieza
df_clean = df_features.copy()

# Registrar operaciones de limpieza
cleaning_log = {
    'timestamp': datetime.now().isoformat(),
    'input_shape': df_features.shape,
    'total_missing_before': int(df_features.isnull().sum().sum()),
    'operations': []
}

print("=" * 80)
print("INICIANDO LIMPIEZA DE NaNs")
print("=" * 80)
print(f"Missing inicial: {df_features.isnull().sum().sum():,} ({(df_features.isnull().sum().sum() / df_features.size * 100):.2f}%)")

INICIANDO LIMPIEZA DE NaNs
Missing inicial: 5,363,614 (8.22%)


### 3.1 Forward Fill para Features Temporales

In [7]:
# Identificar columnas temporales (lags, rolling, CFTC, GDELT, BDI, Crop, Stocks)
temporal_cols = [
    col for col in df_clean.columns 
    if '_lag_' in col or '_roll_' in col or '_ma_' in col or '_ema_' in col
    or col.startswith('cftc_') or col.startswith('gdelt_') or col.startswith('sentiment_')
    or col.startswith('bdi_') or col.startswith('crop_') or col.startswith('gov_stocks_')
]

print(f"\n[1] Forward Fill para {len(temporal_cols)} features temporales")
print(f"    Incluye: lags, rolling, CFTC, GDELT, BDI, Crop, Gov Stocks")
missing_before_ffill = df_clean[temporal_cols].isnull().sum().sum()

# Aplicar ffill
df_clean[temporal_cols] = df_clean[temporal_cols].fillna(method='ffill')

missing_after_ffill = df_clean[temporal_cols].isnull().sum().sum()
print(f"    Missing eliminado: {missing_before_ffill:,} → {missing_after_ffill:,} (Δ = {missing_before_ffill - missing_after_ffill:,})")

# Si quedan NaNs en features académicas (primeros días), usar bfill
if missing_after_ffill > 0:
    print(f"    Aplicando backfill para primeros días...")
    df_clean[temporal_cols] = df_clean[temporal_cols].fillna(method='bfill')
    missing_after_bfill = df_clean[temporal_cols].isnull().sum().sum()
    print(f"    Missing después de bfill: {missing_after_bfill:,}")

# Registrar operación
cleaning_log['operations'].append({
    'step': 1,
    'method': 'forward_fill + backfill',
    'columns': temporal_cols,
    'missing_before': int(missing_before_ffill),
    'missing_after': int(df_clean[temporal_cols].isnull().sum().sum())
})


[1] Forward Fill para 7 features temporales
    Incluye: lags, rolling, CFTC, GDELT, BDI, Crop, Gov Stocks
    Missing eliminado: 6,188 → 460 (Δ = 5,728)
    Aplicando backfill para primeros días...
    Missing después de bfill: 0


C:\Users\trico\AppData\Local\Temp\ipykernel_22576\2323850850.py:14: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_clean[temporal_cols] = df_clean[temporal_cols].fillna(method='ffill')
C:\Users\trico\AppData\Local\Temp\ipykernel_22576\2323850850.py:22: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_clean[temporal_cols] = df_clean[temporal_cols].fillna(method='bfill')


### 3.2 Median Imputation para Features Calculados

In [12]:
# Identificar columnas para median imputation (todo excepto temporales)
median_cols = [col for col in df_clean.columns if col not in temporal_cols]

# Separar columnas numéricas y categóricas
numeric_median_cols = [col for col in median_cols if df_clean[col].dtype in ['float64', 'int64']]
categorical_cols = [col for col in median_cols if df_clean[col].dtype == 'object']

print(f"\n[2] Median Imputation para {len(numeric_median_cols)} features numéricos")
missing_before_median = df_clean[numeric_median_cols].isnull().sum().sum()

# Aplicar median imputation a columnas numéricas
for col in numeric_median_cols:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        
        # Si la columna es toda NaN (edge case), usar 0
        if pd.isna(median_val):
            median_val = 0
            print(f"    WARNING: {col} es toda NaN, usando 0")
        
        df_clean[col].fillna(median_val, inplace=True)

missing_after_median = df_clean[numeric_median_cols].isnull().sum().sum()
print(f"    Missing eliminado: {missing_before_median:,} → {missing_after_median:,} (Δ = {missing_before_median - missing_after_median:,})")

# Aplicar mode imputation a columnas categóricas
if len(categorical_cols) > 0:
    print(f"\n[3] Mode Imputation para {len(categorical_cols)} features categóricos")
    missing_before_cat = df_clean[categorical_cols].isnull().sum().sum()
    
    for col in categorical_cols:
        if df_clean[col].isnull().sum() > 0:
            mode_val = df_clean[col].mode()
            if len(mode_val) > 0:
                df_clean[col].fillna(mode_val[0], inplace=True)
            else:
                df_clean[col].fillna('Unknown', inplace=True)
                print(f"    WARNING: {col} sin moda, usando 'Unknown'")
    
    missing_after_cat = df_clean[categorical_cols].isnull().sum().sum()
    print(f"    Missing eliminado: {missing_before_cat:,} → {missing_after_cat:,} (Δ = {missing_before_cat - missing_after_cat:,})")

# Registrar operación
cleaning_log['operations'].append({
    'step': 2,
    'method': 'median_imputation',
    'columns': numeric_median_cols,
    'missing_before': int(missing_before_median),
    'missing_after': int(missing_after_median)
})

if len(categorical_cols) > 0:
    cleaning_log['operations'].append({
        'step': 3,
        'method': 'mode_imputation',
        'columns': categorical_cols,
        'missing_before': int(missing_before_cat),
        'missing_after': int(missing_after_cat)
    })


[2] Median Imputation para 3227 features numéricos
    Missing eliminado: 0 → 0 (Δ = 0)

[3] Mode Imputation para 2 features categóricos
    Missing eliminado: 20 → 0 (Δ = 20)
    Missing eliminado: 0 → 0 (Δ = 0)

[3] Mode Imputation para 2 features categóricos
    Missing eliminado: 20 → 0 (Δ = 20)


C:\Users\trico\AppData\Local\Temp\ipykernel_22576\532998157.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean[col].fillna(mode_val[0], inplace=True)


### 3.3 Verificación Final

In [13]:
# Verificar que no quedan NaNs
total_missing_final = df_clean.isnull().sum().sum()
print("\n" + "=" * 80)
print("VERIFICACIÓN FINAL")
print("=" * 80)
print(f"Missing después de limpieza: {total_missing_final:,}")
print(f"Shape: {df_clean.shape}")
print(f"Período: {df_clean.index.min()} → {df_clean.index.max()}")

# Assert crítico
assert total_missing_final == 0, f"ERROR: Todavía quedan {total_missing_final:,} NaNs después de limpieza"
print("\n✓ DATASET LIMPIO - Sin missing values")

# Actualizar log
cleaning_log['total_missing_after'] = int(total_missing_final)
cleaning_log['output_shape'] = df_clean.shape


VERIFICACIÓN FINAL
Missing después de limpieza: 0
Shape: (20173, 3236)
Período: 2000-01-03 00:00:00 → 2025-11-10 00:00:00

✓ DATASET LIMPIO - Sin missing values


## 4. Validación de Calidad del Dataset Final

### 4.1 Verificar Continuidad Temporal

In [14]:
# Verificar que no hay gaps en el índice temporal
date_range = pd.date_range(start=df_clean.index.min(), end=df_clean.index.max(), freq='D')
missing_dates = date_range.difference(df_clean.index)

print("=" * 80)
print("VALIDACIÓN DE CONTINUIDAD TEMPORAL")
print("=" * 80)
print(f"Fecha inicial: {df_clean.index.min()}")
print(f"Fecha final: {df_clean.index.max()}")
print(f"Días esperados: {len(date_range):,}")
print(f"Días en dataset: {len(df_clean):,}")
print(f"Fechas faltantes: {len(missing_dates)}")

if len(missing_dates) > 0:
    print(f"\nWARNING: {len(missing_dates)} fechas faltantes:")
    print(missing_dates[:10])  # Mostrar primeras 10
else:
    print("\n✓ Serie temporal continua sin gaps")

VALIDACIÓN DE CONTINUIDAD TEMPORAL
Fecha inicial: 2000-01-03 00:00:00
Fecha final: 2025-11-10 00:00:00
Días esperados: 9,444
Días en dataset: 20,173
Fechas faltantes: 2713

DatetimeIndex(['2000-01-08', '2000-01-09', '2000-01-15', '2000-01-16', '2000-01-22', '2000-01-23', '2000-01-29', '2000-01-30', '2000-02-05', '2000-02-06'], dtype='datetime64[ns]', freq=None)


### 4.2 Verificar Ranges de Variables

In [16]:
# Verificar que no hay valores infinitos o extremos anómalos
print("\n" + "=" * 80)
print("VALIDACIÓN DE RANGES")
print("=" * 80)

# Seleccionar solo columnas numéricas
df_numeric = df_clean.select_dtypes(include=[np.number])

# Infinitos
inf_counts = np.isinf(df_numeric).sum().sum()
print(f"Valores infinitos: {inf_counts}")

if inf_counts > 0:
    inf_cols = df_numeric.columns[np.isinf(df_numeric).any()].tolist()
    print(f"    Columnas con infinitos: {inf_cols[:10]}")  # Primeras 10
else:
    print("    ✓ Sin valores infinitos")


VALIDACIÓN DE RANGES
Valores infinitos: 160882
    Columnas con infinitos: ['ONI_price_to_ma30', 'Crude_Oil_log_return1', 'Crude_Oil_simple_return1', 'Crude_Oil_log_return7', 'Crude_Oil_simple_return7', 'Crude_Oil_log_return30', 'Crude_Oil_simple_return30', 'Crude_Oil_log_return90', 'Crude_Oil_simple_return90', 'Brent_Crude_volume_log_return1']
Valores infinitos: 160882
    Columnas con infinitos: ['ONI_price_to_ma30', 'Crude_Oil_log_return1', 'Crude_Oil_simple_return1', 'Crude_Oil_log_return7', 'Crude_Oil_simple_return7', 'Crude_Oil_log_return30', 'Crude_Oil_simple_return30', 'Crude_Oil_log_return90', 'Crude_Oil_simple_return90', 'Brent_Crude_volume_log_return1']


### 4.3 Verificar Correlación con Targets

In [17]:
# Identificar columnas target (precios base de commodities)
target_candidates = [col for col in df_clean.columns if col in ['soy', 'corn', 'wheat', 'coffee', 'sugar']]

if len(target_candidates) > 0:
    print("\n" + "=" * 80)
    print("CORRELACIÓN CON TARGETS (Precios Base)")
    print("=" * 80)
    
    for target in target_candidates:
        # Calcular correlación con todas las features
        corr_with_target = df_clean.corr()[target].sort_values(ascending=False)
        
        print(f"\n{target.upper()} - Top 10 features más correlacionadas:")
        print(corr_with_target.head(10))
else:
    print("\nWARNING: No se encontraron targets en el dataset")

## 5. Guardar Dataset Final para Modelado

In [18]:
# Guardar dataset limpio
output_path = PROCESSED_DIR / 'features_final_modeling.csv'
df_clean.to_csv(output_path)

print("=" * 80)
print("GUARDANDO DATASET FINAL")
print("=" * 80)
print(f"✓ Dataset guardado: {output_path}")
print(f"  Shape: {df_clean.shape}")
print(f"  Size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")
print(f"  Missing values: 0 (verificado)")

GUARDANDO DATASET FINAL
✓ Dataset guardado: C:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\data\processed\features_final_modeling.csv
  Shape: (20173, 3236)
  Size: 847.03 MB
  Missing values: 0 (verificado)


## 6. Guardar Metadata Completa

In [19]:
# Construir metadata completa del dataset final
metadata = {
    'dataset_name': 'features_final_modeling',
    'creation_date': datetime.now().isoformat(),
    'source_files': [
        'features_step4_climate.csv',
        'cftc/cftc_features_2000_2025.csv',
        'gdelt/sentiment_features_2000_2025.csv',
        'bdi/bdi_features.csv',
        'supply_demand/crop_conditions_all_features.csv',
        'supply_demand/government_stocks_ers_all_features.csv'
    ],
    'cleaning_applied': True,
    
    # Dimensiones
    'shape': {
        'rows': df_clean.shape[0],
        'columns': df_clean.shape[1]
    },
    
    # Período temporal
    'temporal_range': {
        'start': df_clean.index.min().isoformat(),
        'end': df_clean.index.max().isoformat(),
        'days': len(df_clean),
        'missing_dates': len(missing_dates)
    },
    
    # Calidad de datos
    'data_quality': {
        'missing_values': 0,
        'infinite_values': int(inf_counts),
        'total_cells': int(df_clean.size)
    },
    
    # Composición de features
    'feature_composition': {
        'total_features': len(df_clean.columns),
        'temporal_lags': len([c for c in df_clean.columns if '_lag_' in c]),
        'rolling_stats': len([c for c in df_clean.columns if '_roll_' in c or '_ma_' in c or '_ema_' in c]),
        'returns_volatility': len([c for c in df_clean.columns if '_return_' in c or '_vol_' in c or '_bb_' in c]),
        'climate': len([c for c in df_clean.columns if c.startswith('temp_') or c.startswith('prec_') or c in ['oni', 'et0_global', 'gdd_30d_global', 'heat_stress_days_30d', 'prec_deficit_30d']]),
        'cftc_sentiment': len([c for c in df_clean.columns if c.startswith('cftc_')]),
        'gdelt_sentiment': len([c for c in df_clean.columns if c.startswith('gdelt_') or c.startswith('sentiment_')]),
        'bdi': len([c for c in df_clean.columns if c.startswith('bdi_')]),
        'crop_conditions': len([c for c in df_clean.columns if c.startswith('crop_')]),
        'gov_stocks': len([c for c in df_clean.columns if c.startswith('gov_stocks_')]),
        'base': len([c for c in df_clean.columns if c in ['soy', 'corn', 'wheat', 'coffee', 'sugar', 'crude_oil', 'gold', 'usd_index', 'vix', 'sp500']])
    },
    
    # Log de limpieza
    'cleaning_log': cleaning_log,
    
    # Target variables disponibles
    'targets_available': target_candidates,
    
    # Columnas
    'columns': df_clean.columns.tolist()
}

# Guardar metadata
metadata_path = PROCESSED_DIR / 'metadata_final_dataset.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n✓ Metadata guardada: {metadata_path}")


✓ Metadata guardada: C:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\data\processed\metadata_final_dataset.json


## Resumen Final

In [20]:
print("\n" + "=" * 80)
print("RESUMEN - DATASET FINAL PARA MODELADO")
print("=" * 80)

print("\n✓ DATASET LIMPIO Y VALIDADO")
print(f"  - Shape: {df_clean.shape[0]:,} días × {df_clean.shape[1]:,} features")
print(f"  - Período: {df_clean.index.min().date()} → {df_clean.index.max().date()}")
print(f"  - Missing values: 0")
print(f"  - Infinite values: {inf_counts}")

print("\n✓ COMPOSICIÓN DE FEATURES:")
print(f"  - Temporal Lags: {metadata['feature_composition']['temporal_lags']}")
print(f"  - Rolling Stats: {metadata['feature_composition']['rolling_stats']}")
print(f"  - Returns & Volatility: {metadata['feature_composition']['returns_volatility']}")
print(f"  - Climate: {metadata['feature_composition']['climate']}")
print(f"  - CFTC Sentiment: {metadata['feature_composition']['cftc_sentiment']}")
print(f"  - GDELT Sentiment: {metadata['feature_composition']['gdelt_sentiment']}")
print(f"  - Baltic Dry Index: {metadata['feature_composition']['bdi']}")
print(f"  - Crop Conditions: {metadata['feature_composition']['crop_conditions']}")
print(f"  - Government Stocks: {metadata['feature_composition']['gov_stocks']}")
print(f"  - Base Features: {metadata['feature_composition']['base']}")

print("\n✓ LIMPIEZA APLICADA:")
print(f"  - Forward fill + backfill: {cleaning_log['operations'][0]['missing_before'] - cleaning_log['operations'][0]['missing_after']:,} NaNs eliminados")
print(f"  - Median imputation: {cleaning_log['operations'][1]['missing_before'] - cleaning_log['operations'][1]['missing_after']:,} NaNs eliminados")

print("\n✓ ARCHIVOS GENERADOS:")
print(f"  - {output_path.name} ({output_path.stat().st_size / 1024 / 1024:.1f} MB)")
print(f"  - {metadata_path.name}")

print("\n" + "=" * 80)
print("DATASET LISTO PARA NOTEBOOKS DE MODELADO (3.x)")
print("=" * 80)
print("\nPróximos pasos:")
print("  1. Usar features_final_modeling.csv en notebooks 3.1, 3.2, 3.3, 3.4, 3.5")
print("  2. ELIMINAR limpieza de NaNs redundante en cada notebook de modelado")
print("  3. Cargar directamente sin necesidad de fillna/dropna adicional")
print("  4. Comparar Walk-Forward con Step 4 → Step 9 (medir impacto features académicas)")


RESUMEN - DATASET FINAL PARA MODELADO

✓ DATASET LIMPIO Y VALIDADO
  - Shape: 20,173 días × 3,236 features
  - Período: 2000-01-03 → 2025-11-10
  - Missing values: 0
  - Infinite values: 160882

✓ COMPOSICIÓN DE FEATURES:
  - Temporal Lags: 3
  - Rolling Stats: 0
  - Returns & Volatility: 784
  - Climate: 0
  - CFTC Sentiment: 0
  - GDELT Sentiment: 0
  - Baltic Dry Index: 7
  - Crop Conditions: 0
  - Government Stocks: 0
  - Base Features: 0

✓ LIMPIEZA APLICADA:
  - Forward fill + backfill: 6,188 NaNs eliminados
  - Median imputation: 303,017 NaNs eliminados

✓ ARCHIVOS GENERADOS:
  - features_final_modeling.csv (847.0 MB)
  - metadata_final_dataset.json

DATASET LISTO PARA NOTEBOOKS DE MODELADO (3.x)

Próximos pasos:
  1. Usar features_final_modeling.csv en notebooks 3.1, 3.2, 3.3, 3.4, 3.5
  2. ELIMINAR limpieza de NaNs redundante en cada notebook de modelado
  3. Cargar directamente sin necesidad de fillna/dropna adicional
  4. Comparar Walk-Forward con Step 4 → Step 9 (medir imp